# Setup

In [1]:
print("Hello, World!")

Hello, World!


In [2]:
from typing import TypedDict, Annotated, List, Literal
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
import operator
from datetime import datetime
import json


In [3]:
from langchain_openai import ChatOpenAI
import textwrap


# Step 1: Create an LLM client with parameters suitable for your task
llm = ChatOpenAI(
    model="gpt-4o",  # Use the appropriate model for your task
    temperature=0.1, # 0 for deterministic answers, higher for more creative outputs
    max_tokens=None, # More tokens for longer creative content
    timeout=None,
    max_retries=2,
    # other params...
)

# Single agent graph

## State structure

In [6]:
# Define the state structure
class CustomerSupportStateSimple(TypedDict):
    # messages: Annotated[List[dict], operator.add]
    messages: List[dict] 
    customer_query: str
    final_response: str

## Support agent

In [7]:
def support_agent(state: CustomerSupportStateSimple) -> CustomerSupportStateSimple:
    """Analyzes customer query and provide an appropriate response"""
    print("Running support_agent")
    query = state["customer_query"]
    system_prompt = """
    You are Jane Doe, a support agent for customer support.
    Analyze the customer query and determine the query type and urgency level.
    
    Query type can take following values: order_inquiry, technical_support, return_request, general_question.
    Follow these rules strictly:
    - If the query is asking about status of an order, classify the query type as order_inquiry.
    - If the query is asking for technical support or help with a device, classify the query as technical_support.
    - Otherwise, classify the query as general_question. 
    Choose this option only if the query clearly does not match any of the other types.

    Urgency levels can be: low, medium, high

    Assess the sentiment of the customer query:
    1. Emotional tone (positive, neutral, negative, angry)
    2. Frustration level (1-10)
    3. Whether human escalation is recommended

    Create a professional, empathetic response based on all gathered information.
    
    Guidelines:
    - Be friendly and professional
    - Address the customer's specific concern
    - Provide clear solutions or next steps
    - If escalation is needed, mention that a specialist will contact them
    - Keep the response concise but complete
    """
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=f"Customer query: {query}")]
    response = llm.invoke(messages)
    # response = structured_llm.invoke(messages)
    try:
        state["final_response"] = response.content
    except:
        state["final_response"] = "I'm sorry, I couldn't process your request."
    state["messages"].append({"agent": "support_agent", "content": "Final response composed"})
    return state


## Build the graph

In [8]:
# Build the graph
def create_single_agent_support_graph():
    """Creates and returns the customer support agent graph"""
    # Initialize the graph
    graph = StateGraph(CustomerSupportStateSimple)
    # Add nodes
    graph.add_node("support", support_agent)
    # Add edges
    graph.set_entry_point("support")
    # After support agent execution ends
    graph.add_edge("support", END)
    return graph.compile(debug=False)

## Demonstration of the single-agent network

In [9]:
"""Runs demo scenarios through the support system"""
# Create the graph
simple_support_graph = create_single_agent_support_graph()
# Test scenarios
test_queries = [
    "I have placed an order and I need to check the status of my order, the order ID is ORD-12345",
    "My laptop won't turn on after the latest update. This is so frustrating!",
    "I need technical support, how can I update the software on the device?",
    "What's your return policy? I might need to return my monitor.",
    "I forgot my password and can't log into my account",
    "Order  ORD-67890was supposed to arrive yesterday but it didn't! This is unacceptable!"
]
print("=== Single Agent Support Demo ===\n")

for i, query in enumerate(test_queries, 1):
    print(f"Scenario {i}: {query}")
    print("-" * 50)
    
    # Initialize state
    initial_state = CustomerSupportStateSimple(
        messages=[],
        customer_query=query,
        final_response=""
    )
    # Run the graph
    final_state = simple_support_graph.invoke(initial_state)

    # Display results
    print("\n=== Final State ===")
    print(f"\nAgent Activity:")
    for msg in final_state['messages']:
        print(f"  - {msg['agent']}: {msg['content']}")
    print(f"\nFinal Response:")
    print(final_state['final_response'])
    print("\n" + "="*50 + "\n")



=== Single Agent Support Demo ===

Scenario 1: I have placed an order and I need to check the status of my order, the order ID is ORD-12345
--------------------------------------------------
Running support_agent

=== Final State ===

Agent Activity:
  - support_agent: Final response composed

Final Response:
**Query Type:** order_inquiry  
**Urgency Level:** medium  

**Sentiment Analysis:**  
1. Emotional tone: neutral  
2. Frustration level: 2  
3. Human escalation: Not recommended  

**Response:**

Hello,

Thank you for reaching out to us. I understand that you would like to check the status of your order with the ID ORD-12345. I’m here to help you with that.

Please allow me a moment to look into your order details. [Assuming access to order system, provide status here, e.g., "Your order is currently being processed and is expected to ship within the next 2-3 business days."]

If you have any further questions or need additional assistance, feel free to let me know. I'm here to he

## Single agent graph structure

In [10]:
# Graph structure
print("\n\n GRAPH STRUCTURE")
print("-" * 50)
graph_structure = simple_support_graph.get_graph()
print(f"Nodes: {list(graph_structure.nodes.keys())}")
print(f"\nEdges:")
for edge in graph_structure.edges:
    print(f"  {edge}")



 GRAPH STRUCTURE
--------------------------------------------------
Nodes: ['__start__', 'support', '__end__']

Edges:
  Edge(source='__start__', target='support', data=None, conditional=False)
  Edge(source='support', target='__end__', data=None, conditional=False)


# Multi-agent graph

## Define the state structure

In [11]:
# Define the state structure
class CustomerSupportState(TypedDict):
    # messages: Annotated[List[dict], operator.add]
    messages: List[dict] 
    customer_query: str
    query_type: str
    sentiment: str
    order_id: str
    technical_issue: str
    knowledge_base_results: List[str]
    proposed_solution: str
    requires_escalation: bool
    final_response: str

## Mock data

### Orders database

In [12]:
# Mock data for demonstration
MOCK_ORDERS = {
    "ORD-12345": {
        "status": "Delivered",
        "date": "2024-03-15",
        "items": ["Laptop", "Mouse"],
        "tracking": "TRK-98765"
    },
    "ORD-67890": {
        "status": "In Transit",
        "date": "2024-03-18",
        "items": ["Monitor"],
        "tracking": "TRK-54321"
    }
}

### Knowledge base (How-to/FAQ documents)

In [13]:
MOCK_KNOWLEDGE_BASE = {
    "password_reset": "To reset your password: 1) Click 'Forgot Password' on login page 2) Enter your email 3) Check your email for reset link 4) Create new password",
    "return_policy": "Items can be returned within 30 days of delivery. Items must be unused and in original packaging. Refunds processed within 5-7 business days.",
    "shipping_delays": "Current estimated delivery times: Standard (5-7 days), Express (2-3 days). Weather conditions may cause delays.",
    "technical_support": "For technical issues: 1) Restart your device 2) Check for updates 3) Clear cache 4) Contact support if issue persists"
}

## Router agent

In [14]:
def router_agent(state: CustomerSupportState) -> CustomerSupportState:
    """Analyzes customer query and routes to appropriate agents"""
    print("Running router_agent")
    query = state["customer_query"]
    system_prompt = """You are a routing agent for customer support. 
    Analyze the customer query and determine the query type and urgency level.
    
    Query type can take following values: order_inquiry, technical_support, return_request, general_question.
    Follow these rules strictly:
    - If the query is asking about status of an order, classify the query type as order_inquiry.
    - If the query is asking for technical support or help with a device, classify the query as technical_support.
    - Otherwise, classify the query as general_question. Choose this option only if the query clearly does not match any of the other types.

    Urgency levels can be: low, medium, high

    If an order ID is mentioned (format: ORD-XXXXX)n, extract to the field order_id. If no order ID, set the order_id to an empty string.

    Respond in JSON format: {"query_type": "...", "order_id": "...", "urgency": "low/medium/high"}
    Only return the JSON response without any additional text.
    """
    messages = [
        SystemMessage(content=system_prompt), HumanMessage(content=f"Customer query: {query}")
    ]
    response = llm.invoke(messages)
    try:
        analysis = json.loads(response.content)
        state["query_type"] = analysis.get("query_type", "general_question")
        state["order_id"] = analysis.get("order_id", "")
        state["urgency"] = analysis.get("urgency", "low")
    except:
        state["query_type"] = "general_question"
        state["order_id"] = ""
        state["urgency"] = "low"
    state["messages"].append({"agent": "router","content": f"Classified as: {state['query_type']}"})
    return state


## Sentiment Analysis Agent

In [15]:
def sentiment_analysis_agent(state: CustomerSupportState) -> CustomerSupportState:
    """Analyzes customer sentiment and determines if escalation is needed"""
    print("Running sentiment_analysis_agent")
    query = state["customer_query"]
    system_prompt = """You are a sentiment analysis agent. 
    Analyze the customer's message for:
    1. Emotional tone (positive, neutral, negative, angry)
    2. Frustration level (1-10)
    3. Whether human escalation is recommended 
    Respond in JSON format: {"sentiment": "...", "frustration_level": X, "escalate": true/false}""" 
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Customer message: {query}")
    ] 
    response = llm.invoke(messages)
    try:
        analysis = json.loads(response.content)
        state["sentiment"] = analysis.get("sentiment", "neutral")
        state["requires_escalation"] = analysis.get("escalate", False)
    except:
        state["sentiment"] = "neutral"
        state["requires_escalation"] = False
    
    state["messages"].append({"agent": "sentiment_analysis",
        "content": f"Sentiment: {state['sentiment']}, Escalation: {state['requires_escalation']}"
    })
    return state

## Knowledge base Agent

In [16]:
def knowledge_base_agent(state: CustomerSupportState) -> CustomerSupportState:
    """Searches knowledge base for relevant information"""
    print("Running knowledge_base_agent")
    query = state["customer_query"]
    query_type = state["query_type"]
    system_prompt = """You are a knowledge base agent.
    Search for relevant information based on the query and type.
    List the most relevant knowledge base articles or solutions.
    """
    # Mock knowledge base search
    relevant_articles = []
    query_lower = query.lower()
    for key, content in MOCK_KNOWLEDGE_BASE.items():
        if any(word in query_lower for word in key.split("_")):
            relevant_articles.append(content)
    state["knowledge_base_results"] = relevant_articles
    state["messages"].append({"agent": "knowledge_base", "content": f"Found {len(relevant_articles)} relevant articles"})
    return state

## Order management agent

In [17]:
def order_management_agent(state: CustomerSupportState) -> CustomerSupportState:
    """Handles order-related queries"""
    print("Running order_management_agent")
    order_id = state["order_id"]
    if not order_id:
        state["messages"].append({"agent": "order_management","content": "No order ID provided"})
        return state
    order_info = MOCK_ORDERS.get(order_id)
    if order_info:
        order_details = f"Order {order_id}: Status - {order_info['status']}, " \
                       f"Shipped on {order_info['date']}, " \
                       f"Tracking: {order_info['tracking']}"
        
        state["messages"].append({"agent": "order_management","content": order_details})
        # Add to proposed solution
        state["proposed_solution"] = order_details
    else:
        state["messages"].append({"agent": "order_management","content": f"Order {order_id} not found"})
    return state

## Technical support agent

In [18]:
def technical_support_agent(state: CustomerSupportState) -> CustomerSupportState:
    """Handles technical support queries"""
    print("Running technical_support_agent")
    if state["query_type"] != "technical_support":
        return state
    query = state["customer_query"]
    system_prompt = """You are a technical support agent.
    Based on the customer's technical issue:
    1. Identify the specific problem
    2. Suggest troubleshooting steps
    3. Determine if remote assistance is needed  
    Provide clear, step-by-step solutions."""
    
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=f"Technical issue: {query}")]
    response = llm.invoke(messages)
    state["technical_issue"] = response.content
    state["messages"].append({"agent": "technical_support","content": "Technical diagnosis completed"})
    return state

## Response composer agent

In [19]:
def response_composer_agent(state: CustomerSupportState) -> CustomerSupportState:
    """Composes the final response to the customer"""
    print("Running response_composer_agent")
    # Gather all information
    context = {
        "query": state["customer_query"],
        "query_type": state["query_type"],
        "sentiment": state["sentiment"],
        "knowledge_base": state["knowledge_base_results"],
        "order_info": state.get("proposed_solution", ""),
        "technical_info": state.get("technical_issue", ""),
        "requires_escalation": state["requires_escalation"]
    }
    system_prompt = """You are John Doe, a response composition agent for customer support.
    Create a professional, empathetic response based on all gathered information.
    Guidelines:
    - Be friendly and professional
    - Address the customer's specific concern
    - Provide clear solutions or next steps
    - If escalation is needed, mention that a specialist will contact them
    - Keep the response concise but complete
    """
    
    messages = [SystemMessage(content=system_prompt),HumanMessage(content=f"Context: {json.dumps(context, indent=2)}")]
    response = llm.invoke(messages)  
    state["final_response"] = response.content
    state["messages"].append({"agent": "response_composer", "content": "Final response composed"})
    return state

## Logic for conditional edges

In [20]:
# Define conditional logic

def should_check_order_or_technical(state: CustomerSupportState) -> Literal["order", "technical", "skip"]:
    """Determines if order management or technical support agent should be called"""
    if state["order_id"] or state["query_type"] == "order_inquiry":
        return "order"
    elif state["query_type"] == "technical_support":
        return "technical"
    return "skip"

def should_escalate(state: CustomerSupportState) -> Literal["escalate", "continue"]:
    """Determines if human escalation is needed"""
    if state["requires_escalation"]:
        return "escalate"
    return "continue"

## Creating the multi-agent graph

In [21]:
# Build the graph
def create_support_graph():
    """Creates and returns the customer support agent graph"""
    # Initialize the graph
    graph = StateGraph(CustomerSupportState)
    # Add nodes
    graph.add_node("router", router_agent)
    graph.add_node("sentiment_analysis", sentiment_analysis_agent)
    graph.add_node("knowledge_base", knowledge_base_agent)
    graph.add_node("order_management", order_management_agent)
    graph.add_node("technical_support", technical_support_agent)
    graph.add_node("response_composer", response_composer_agent)
    # Add edges
    graph.set_entry_point("router") 
    # Router connects to sentiment analysis
    graph.add_edge("router", "sentiment_analysis")
    # Sentiment connects to knowledge base
    graph.add_edge("sentiment_analysis", "knowledge_base")
    graph.add_conditional_edges(
        "knowledge_base",
        should_check_order_or_technical,
        {
            "order": "order_management",
            "technical": "technical_support",
            "skip": "response_composer"
        }
    )
    # Order management connects to response composer
    graph.add_edge("order_management", "response_composer")
    # Technical support connects to response composer
    graph.add_edge("technical_support", "response_composer")
    # Response composer can escalate or end
    graph.add_conditional_edges("response_composer",should_escalate,{"escalate": END,"continue": END})
    return graph.compile(debug=False)

## Demonstration of the multi-agent network

In [22]:
"""Runs demo scenarios through the support system"""

# Create the graph
support_graph = create_support_graph()

# Test scenarios
test_queries = [
    "I have placed an order and I need to check the status of my order, the order ID is ORD-12345",
    "My laptop won't turn on after the latest update. This is so frustrating!",
    "I need technical support, how can I update the software on the device?",
    "What's your return policy? I might need to return my monitor.",
    "I forgot my password and can't log into my account",
    "Order ORD-67890 was supposed to arrive yesterday but it didn't! This is unacceptable!"
]

print("=== Customer Support Agent Network Demo ===\n")
for i, query in enumerate(test_queries, 1):
    print(f"Scenario {i}: {query}")
    print("-" * 50)
    
    # Initialize state
    initial_state = {
        "messages": [],
        "customer_query": query,
        "query_type": "",
        "sentiment": "",
        "order_id": "",
        "technical_issue": "",
        "knowledge_base_results": [],
        "proposed_solution": "",
        "requires_escalation": False,
        "final_response": ""
    }
    
    # Run the graph
    final_state = support_graph.invoke(initial_state)
    
    # Display results
    print("\n=== Final State ===")
    print(f"Query Type: {final_state['query_type']}")
    print(f"Sentiment: {final_state['sentiment']}")
    print(f"Escalation Required: {final_state['requires_escalation']}")
    print(f"\nAgent Activity:")
    for msg in final_state['messages']:
        print(f"  - {msg['agent']}: {msg['content']}")
    print(f"\nFinal Response:")
    print(final_state['final_response'])
    print("\n" + "="*50 + "\n")



=== Customer Support Agent Network Demo ===

Scenario 1: I have placed an order and I need to check the status of my order, the order ID is ORD-12345
--------------------------------------------------
Running router_agent
Running sentiment_analysis_agent
Running knowledge_base_agent
Running order_management_agent
Running response_composer_agent

=== Final State ===
Query Type: order_inquiry
Sentiment: neutral
Escalation Required: False

Agent Activity:
  - router: Classified as: order_inquiry
  - sentiment_analysis: Sentiment: neutral, Escalation: False
  - knowledge_base: Found 0 relevant articles
  - order_management: Order ORD-12345: Status - Delivered, Shipped on 2024-03-15, Tracking: TRK-98765
  - response_composer: Final response composed

Final Response:
Dear Customer,

Thank you for reaching out to us regarding your order status. I’m happy to assist you with your inquiry.

I have checked the details for your order with ID ORD-12345. I’m pleased to inform you that your order has

## Graph structure

In [23]:
# Graph structure
print("\n\n3. GRAPH STRUCTURE")
print("-" * 50)
graph_structure = support_graph.get_graph()
print(f"Nodes: {list(graph_structure.nodes.keys())}")
print(f"\nEdges:")
for edge in graph_structure.edges:
    print(f"  {edge}")



3. GRAPH STRUCTURE
--------------------------------------------------
Nodes: ['__start__', 'router', 'sentiment_analysis', 'knowledge_base', 'order_management', 'technical_support', 'response_composer', '__end__']

Edges:
  Edge(source='__start__', target='router', data=None, conditional=False)
  Edge(source='knowledge_base', target='order_management', data='order', conditional=True)
  Edge(source='knowledge_base', target='response_composer', data='skip', conditional=True)
  Edge(source='knowledge_base', target='technical_support', data='technical', conditional=True)
  Edge(source='order_management', target='response_composer', data=None, conditional=False)
  Edge(source='response_composer', target='__end__', data='continue', conditional=True)
  Edge(source='router', target='sentiment_analysis', data=None, conditional=False)
  Edge(source='sentiment_analysis', target='knowledge_base', data=None, conditional=False)
  Edge(source='technical_support', target='response_composer', data=No